# Rankings Integration: World Football Elo + FIFA World Rankings

This notebook integrates two external team-strength signals into the processed ML dataset:

| Source | Coverage | Key Feature |
|--------|----------|-------------|
| **World Football Elo** (eloratings.net) | 1901–2023 | `home/away_elo_rating` |
| **FIFA World Rankings** (Kaggle / fifa.com) | Dec 1992–Jun 2024 | `home/away_fifa_rank`, `home/away_fifa_points` |

**Leakage prevention rule:** For each World Cup year, we use the rating/ranking snapshot from *before* the tournament started — never the end-of-year value.

In [1]:
import pandas as pd
import numpy as np

# Paths
DATA_DIR = "../../data"
ELO_PATH = f"{DATA_DIR}/raw/elo_ratings.csv"
FIFA_PATH = f"{DATA_DIR}/raw/fifa_ranking-2024-06-20.csv"
TEAMS_PATH = f"{DATA_DIR}/raw/teams.csv"
MATCHES_PATH = f"{DATA_DIR}/raw/match_features_base.csv"
PROCESSED_PATH = f"{DATA_DIR}/processed/class_a_match_level.csv"
OUTPUT_PATH = f"{DATA_DIR}/processed/class_a_with_rankings.csv"

## Team ID to team name mapping

The processed ML dataset uses team IDs (e.g. `T-31`). We need full names to join against Elo and FIFA data.

We use the **official `teams.csv`** from the [jfjelstul/worldcup](https://github.com/jfjelstul/worldcup) R package. This covers all 88 team IDs across all World Cups from 1930–2022.

In [2]:
# Load official team mapping
teams_df = pd.read_csv(TEAMS_PATH)
team_id_to_name = teams_df.set_index("team_id")["team_name"].to_dict()

# Verify all team IDs in processed dataset are covered
processed_check = pd.read_csv(PROCESSED_PATH, usecols=["home_team_id", "away_team_id"])
all_ids = set(processed_check["home_team_id"]) | set(processed_check["away_team_id"])
unmapped = all_ids - set(team_id_to_name.keys())

print(f"Total teams in official CSV: {len(team_id_to_name)}")
print(f"Unique team IDs in processed data: {len(all_ids)}")

sample_ids = ["T-05", "T-07", "T-10", "T-21", "T-60", "T-61", "T-72", "T-87"]
print("\nSample pre-2006 mappings from official CSV:")
for tid in sample_ids:
    print(f"  {tid} is {team_id_to_name.get(tid, 'NOT FOUND')}")

Total teams in official CSV: 88
Unique team IDs in processed data: 88

Sample pre-2006 mappings from official CSV:
  T-05 is Austria
  T-07 is Bolivia
  T-10 is Bulgaria
  T-21 is Czechoslovakia
  T-60 is Republic of Ireland
  T-61 is Romania
  T-72 is Soviet Union
  T-87 is Yugoslavia


## Load & Prepare World Football Elo Ratings

Source: eloratings.net 

In [3]:
elo = pd.read_csv(ELO_PATH)
elo["year"] = elo["year"].astype(int)

# Keep only the columns we need
elo = elo[["year", "team", "rating", "rank"]].copy()
elo.columns = ["elo_year", "team_name", "elo_rating", "elo_rank"]

# The join key elo_year = tournament_year - 1 (pre-tournament snapshot)
elo["year"] = elo["elo_year"] + 1

# Handle duplicate ratings (keep highest rated entry per team per year)
elo = elo.sort_values("elo_rating", ascending=False).drop_duplicates(
    subset=["year", "team_name"]
)

print(f"Elo records: {len(elo):,}")
print(
    f"Tournament years covered (via shift): {elo['year'].min()} - {elo['year'].max()}"
)
print(elo.head())

Elo records: 17,200
Tournament years covered (via shift): 1902 - 2024
      elo_year team_name  elo_rating  elo_rank  year
3273      1955   Hungary        2213         1  1956
165       1912   England        2212         1  1913
99        1909   England        2207         1  1910
3138      1954   Hungary        2192         1  1955
4288      1962    Brazil        2192         1  1963


### Team Name Normalization for Elo

Map project team namesto Elo dataset team names to handle discrepancies.

In [4]:
# Elo dataset team name
ELO_NAME_MAP = {
    "United States": "United States",
    "North Korea": "North Korea",
    "Dutch East Indies": "Indonesia",
    "Bosnia and Herzegovina": "Bosnia-Herzegovina",
}


def normalize_for_elo(name):
    return ELO_NAME_MAP.get(name, name)


# Verify key teams exist in Elo data
sample_teams = ["France", "Brazil", "Germany", "South Korea", "Ivory Coast", "Iran"]
print("Elo coverage check:")
for t in sample_teams:
    found = t in elo["team_name"].values
    print(f"  {t}: {'FOUND' if found else 'MISSING'}")

Elo coverage check:
  France: FOUND
  Brazil: FOUND
  Germany: FOUND
  South Korea: FOUND
  Ivory Coast: FOUND
  Iran: FOUND


## Load & Prepare FIFA World Rankings

Kaggle (cashncarry/fifaworldranking) 

For each World Cup year, use the **last ranking snapshot published before the tournament kick-off**.

In [5]:
fifa = pd.read_csv(FIFA_PATH)
fifa["rank_date"] = pd.to_datetime(fifa["rank_date"])
print(f"FIFA ranking records: {len(fifa):,}")
print(
    f"Date range: {fifa['rank_date'].min().date()} to {fifa['rank_date'].max().date()}"
)
print(f"Unique countries: {fifa['country_full'].nunique()}")
print(fifa.head())

FIFA ranking records: 67,472
Date range: 1992-12-31 to 2024-06-20
Unique countries: 216
    rank       country_full country_abrv  total_points  previous_points  \
0  140.0  Brunei Darussalam          BRU           2.0              0.0   
1   33.0           Portugal          POR          38.0              0.0   
2   32.0             Zambia          ZAM          38.0              0.0   
3   31.0             Greece          GRE          38.0              0.0   
4   30.0            Algeria          ALG          39.0              0.0   

   rank_change confederation  rank_date  
0          140           AFC 1992-12-31  
1           33          UEFA 1992-12-31  
2           32           CAF 1992-12-31  
3           31          UEFA 1992-12-31  
4           30           CAF 1992-12-31  


In [6]:
# World Cup start dates
## For each World Cup year, we use the rating/ranking snapshot from before the tournament started, never the end-of-year value.
WC_START_DATES = {
    1994: "1994-06-17",
    1998: "1998-06-10",
    2002: "2002-05-31",
    2006: "2006-06-09",
    2010: "2010-06-11",
    2014: "2014-06-12",
    2018: "2018-06-14",
    2022: "2022-11-20",
}

fifa_snapshots = []
for wc_year, start_date in WC_START_DATES.items():
    cutoff = pd.Timestamp(start_date)
    pre_wc = fifa[fifa["rank_date"] < cutoff]
    if pre_wc.empty:
        print(f"WARNING: No FIFA snapshot found before {wc_year} WC")
        continue
    snapshot_date = pre_wc["rank_date"].max()
    snapshot = pre_wc[pre_wc["rank_date"] == snapshot_date].copy()
    snapshot["wc_year"] = wc_year
    fifa_snapshots.append(snapshot)
    print(
        f"{wc_year} WC: using FIFA snapshot from {snapshot_date.date()} ({len(snapshot)} teams)"
    )

fifa_pre_wc = pd.concat(fifa_snapshots, ignore_index=True)
fifa_pre_wc = fifa_pre_wc[["wc_year", "country_full", "rank", "total_points"]].copy()
fifa_pre_wc.columns = ["year", "team_name", "fifa_rank", "fifa_points"]
print(f"\nTotal FIFA pre-WC records: {len(fifa_pre_wc):,}")

1994 WC: using FIFA snapshot from 1994-06-14 (172 teams)
1998 WC: using FIFA snapshot from 1998-05-20 (191 teams)
2002 WC: using FIFA snapshot from 2002-05-15 (202 teams)
2006 WC: using FIFA snapshot from 2006-05-17 (204 teams)
2010 WC: using FIFA snapshot from 2010-05-26 (208 teams)
2014 WC: using FIFA snapshot from 2014-06-05 (209 teams)
2018 WC: using FIFA snapshot from 2018-06-07 (211 teams)
2022 WC: using FIFA snapshot from 2022-10-06 (211 teams)

Total FIFA pre-WC records: 1,608


### Team Name Normalization for FIFA



In [7]:
# FIFA country_full name
FIFA_NAME_MAP = {
    "United States": "USA",
    "South Korea": "Korea Republic",
    "North Korea": "Korea DPR",
    "Ivory Coast": "Côte d'Ivoire",
    "Iran": "IR Iran",
    "Serbia and Montenegro": "Serbia and Montenegro",
    "Czech Republic": "Czech Republic",
    "Republic of Ireland": "Ireland",
    "China": "China PR",
    "Bosnia and Herzegovina": "Bosnia-Herzegovina",
    "Serbia": "Serbia",
}


def normalize_for_fifa(name):
    return FIFA_NAME_MAP.get(name, name)


# Verify key teams exist in FIFA data
sample_checks = [
    "USA",
    "Korea Republic",
    "Côte d'Ivoire",
    "France",
    "Brazil",
    "IR Iran",
]
print("FIFA coverage check:")
for t in sample_checks:
    found = t in fifa_pre_wc["team_name"].values
    print(f"  {t}: {'FOUND' if found else 'MISSING'}")

FIFA coverage check:
  USA: FOUND
  Korea Republic: FOUND
  Côte d'Ivoire: FOUND
  France: FOUND
  Brazil: FOUND
  IR Iran: FOUND


## Load Processed ML Dataset & Add Team Names

In [8]:
processed = pd.read_csv(PROCESSED_PATH)
print(f"Processed dataset shape: {processed.shape}")
print(f"Year range: {processed['year'].min()} - {processed['year'].max()}")

# Map team IDs to names
processed["home_team_name"] = processed["home_team_id"].map(team_id_to_name)
processed["away_team_name"] = processed["away_team_id"].map(team_id_to_name)

# Report any still-unmapped teams
unmapped_home = processed[processed["home_team_name"].isna()]["home_team_id"].unique()
unmapped_away = processed[processed["away_team_name"].isna()]["away_team_id"].unique()
all_unmapped = set(unmapped_home) | set(unmapped_away)

Processed dataset shape: (1248, 104)
Year range: 1930 - 2022


## Join Elo Ratings

In [9]:
# Normalize team names for Elo join
processed["home_name_elo"] = processed["home_team_name"].apply(normalize_for_elo)
processed["away_name_elo"] = processed["away_team_name"].apply(normalize_for_elo)

# Join home team Elo
elo_lookup = elo[["year", "team_name", "elo_rating", "elo_rank"]]

processed = processed.merge(
    elo_lookup.rename(
        columns={
            "team_name": "home_name_elo",
            "elo_rating": "home_elo_rating",
            "elo_rank": "home_elo_rank",
        }
    ),
    on=["year", "home_name_elo"],
    how="left",
)

# Join away team Elo
processed = processed.merge(
    elo_lookup.rename(
        columns={
            "team_name": "away_name_elo",
            "elo_rating": "away_elo_rating",
            "elo_rank": "away_elo_rank",
        }
    ),
    on=["year", "away_name_elo"],
    how="left",
)

# Derived feature: difference in Elo (home perspective)
processed["feat_elo_rating_diff"] = (
    processed["home_elo_rating"] - processed["away_elo_rating"]
)

elo_coverage = processed["home_elo_rating"].notna().mean()
print(f"Elo coverage: {elo_coverage:.1%} of matches have home Elo rating")
print(
    processed[
        ["year", "home_elo_rating", "away_elo_rating", "feat_elo_rating_diff"]
    ].head(10)
)

Elo coverage: 99.0% of matches have home Elo rating
   year  home_elo_rating  away_elo_rating  feat_elo_rating_diff
0  1930           1538.0           1613.0                 -75.0
1  1930           1673.0           1636.0                  37.0
2  1930           1608.0           1929.0                -321.0
3  1930           1560.0           1542.0                  18.0
4  1930           2084.0           1538.0                 546.0
5  1930           1578.0           1613.0                 -35.0
6  1930           1608.0           1258.0                 350.0
7  1930           1673.0           1784.0                -111.0
8  1930           1963.0           1542.0                 421.0
9  1930           1578.0           1538.0                  40.0


/var/folders/yp/y7bq_swn0clf6763v16fcjhw0000gn/T/ipykernel_14822/1690139011.py:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  processed["feat_elo_rating_diff"] = (


## Join FIFA Rankings

In [10]:
# Normalize team names for FIFA join
processed["home_name_fifa"] = processed["home_team_name"].apply(normalize_for_fifa)
processed["away_name_fifa"] = processed["away_team_name"].apply(normalize_for_fifa)

fifa_lookup = fifa_pre_wc[["year", "team_name", "fifa_rank", "fifa_points"]]

# Join home team FIFA
processed = processed.merge(
    fifa_lookup.rename(
        columns={
            "team_name": "home_name_fifa",
            "fifa_rank": "home_fifa_rank",
            "fifa_points": "home_fifa_points",
        }
    ),
    on=["year", "home_name_fifa"],
    how="left",
)

# Join away team FIFA
processed = processed.merge(
    fifa_lookup.rename(
        columns={
            "team_name": "away_name_fifa",
            "fifa_rank": "away_fifa_rank",
            "fifa_points": "away_fifa_points",
        }
    ),
    on=["year", "away_name_fifa"],
    how="left",
)

# Derived features: rank difference (lower = better, so away - home = positive means home is better ranked)
processed["feat_fifa_rank_diff"] = (
    processed["away_fifa_rank"] - processed["home_fifa_rank"]
)
processed["feat_fifa_points_diff"] = (
    processed["home_fifa_points"] - processed["away_fifa_points"]
)

fifa_coverage = processed["home_fifa_rank"].notna().mean()
print(f"FIFA coverage: {fifa_coverage:.1%} of matches have home FIFA rank")
print("\nFIFA NaN by year (expected NaN for pre-1994 years):")
print(
    processed.groupby("year")["home_fifa_rank"]
    .apply(lambda x: x.isna().sum())
    .rename("missing_fifa")
    .to_string()
)

FIFA coverage: 39.7% of matches have home FIFA rank

FIFA NaN by year (expected NaN for pre-1994 years):
year
1930    18
1934    17
1938    18
1950    22
1954    26
1958    35
1962    32
1966    32
1970    32
1974    38
1978    38
1982    52
1986    52
1990    52
1991    26
1994     1
1995    26
1998     0
1999    32
2002     1
2003    32
2006     2
2007    32
2010     0
2011    32
2014     1
2015    52
2018     0
2019    52
2022     0


/var/folders/yp/y7bq_swn0clf6763v16fcjhw0000gn/T/ipykernel_14822/1029972037.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  processed["home_name_fifa"] = processed["home_team_name"].apply(normalize_for_fifa)
/var/folders/yp/y7bq_swn0clf6763v16fcjhw0000gn/T/ipykernel_14822/1029972037.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  processed["away_name_fifa"] = processed["away_team_name"].apply(normalize_for_fifa)
/var/folders/yp/y7bq_swn0clf6763v16fcjhw0000gn/T/ipykernel_14822/1029972037.py:34: PerformanceWarning: DataFra

## Coverage Summary

In [11]:
new_features = [
    "home_elo_rating",
    "away_elo_rating",
    "home_elo_rank",
    "away_elo_rank",
    "feat_elo_rating_diff",
    "home_fifa_rank",
    "away_fifa_rank",
    "home_fifa_points",
    "away_fifa_points",
    "feat_fifa_rank_diff",
    "feat_fifa_points_diff",
]

print("New feature coverage (non-null %):\n")
coverage = processed[new_features].notna().mean().mul(100).round(1)
print(coverage.to_string())

print(f"\nFull dataset shape: {processed.shape}")
print(f"Original columns: 104  →  New columns: {processed.shape[1]}")
print(f"New features added: {processed.shape[1] - 104}")

New feature coverage (non-null %):

home_elo_rating          99.0
away_elo_rating          99.0
home_elo_rank            99.0
away_elo_rank            99.0
feat_elo_rating_diff     98.1
home_fifa_rank           39.7
away_fifa_rank           39.3
home_fifa_points         39.7
away_fifa_points         39.3
feat_fifa_rank_diff      38.9
feat_fifa_points_diff    38.9

Full dataset shape: (1248, 121)
Original columns: 104  →  New columns: 121
New features added: 17


In [12]:
# Drop the temporary name columns used for joining
cols_to_drop = [
    "home_team_name",
    "away_team_name",
    "home_name_elo",
    "away_name_elo",
    "home_name_fifa",
    "away_name_fifa",
]
processed = processed.drop(columns=cols_to_drop)

# Save
processed.to_csv(OUTPUT_PATH, index=False)
print(f"Saved to: {OUTPUT_PATH}")
print(f"Final shape: {processed.shape}")

# check on 2022 WC
print("\nSanity check — 2022 WC sample:")
cols_check = [
    "year",
    "home_team_id",
    "away_team_id",
    "home_elo_rating",
    "away_elo_rating",
    "feat_elo_rating_diff",
    "home_fifa_rank",
    "away_fifa_rank",
    "feat_fifa_rank_diff",
]
print(processed[processed["year"] == 2022][cols_check].head(8).to_string())

Saved to: ../../data/processed/class_a_with_rankings.csv
Final shape: (1248, 115)

Sanity check — 2022 WC sample:
      year home_team_id away_team_id  home_elo_rating  away_elo_rating  feat_elo_rating_diff  home_fifa_rank  away_fifa_rank  feat_fifa_rank_diff
1184  2022         T-59         T-25           1663.0           1849.0                -186.0            50.0            44.0                 -6.0
1185  2022         T-28         T-38           2032.0           1836.0                 196.0             5.0            20.0                 15.0
1186  2022         T-65         T-48           1705.0           1929.0                -224.0            18.0             8.0                -10.0
1187  2022         T-83         T-85           1860.0           1836.0                  24.0            16.0            19.0                  3.0
1188  2022         T-03         T-63           2101.0           1628.0                 473.0             3.0            51.0                 48.0
1189  2022

## Year Range Filtering 

In [13]:
YEAR_START = 2006
YEAR_END = 2022

filtered = processed[
    (processed["year"] >= YEAR_START) & (processed["year"] <= YEAR_END)
]
print(f"Filtered dataset: {filtered.shape[0]} matches ({YEAR_START}-{YEAR_END})")
print(f"Columns: {filtered.shape[1]}")
filtered.to_csv(
    f"{DATA_DIR}/final/class_a_with_rankings_{YEAR_START}_{YEAR_END}.csv", index=False
)
print(f"Saved: class_a_with_rankings_{YEAR_START}_{YEAR_END}.csv")

Filtered dataset: 488 matches (2006-2022)
Columns: 115
Saved: class_a_with_rankings_2006_2022.csv
